# Team 1 – AI-Enhanced Network Anomaly Detection

## Session 03: Baseline Model Development

**Author:** Jennifer Silva  
**Date:** October 6, 2026

### Purpose

This notebook establishes the baseline machine-learning models for the CICIDS2017 intrusion-detection dataset. It builds on the team's completed data audit and preprocessing work by using the approved training and validation datasets.

The initial models evaluated are:

- Decision Tree
- Random Forest

The models will be trained using the 70% training dataset and evaluated against the 15% validation dataset. Performance will be compared using precision, recall, F1-score, confusion matrices, false-positive behavior, and false-negative behavior.

The separate 15% test dataset will remain untouched during model development and will be reserved for final evaluation.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
DATA_DIR = Path("../data/processed")

TRAIN_FILE = DATA_DIR / "training_dataset_CIC_70.csv"
VAL_FILE = DATA_DIR / "validation_dataset_CIC_15.csv"

print("Training dataset:", TRAIN_FILE)
print("Validation dataset:", VAL_FILE)

assert TRAIN_FILE.exists(), f"Training dataset not found: {TRAIN_FILE}"
assert VAL_FILE.exists(), f"Validation dataset not found: {VAL_FILE}"

print("\nTraining and validation datasets found successfully.")

Training dataset: ../data/processed/training_dataset_CIC_70.csv
Validation dataset: ../data/processed/validation_dataset_CIC_15.csv

Training and validation datasets found successfully.


## 1. Load and Validate Team Datasets

The training and validation datasets were generated during the team's preprocessing phase using a stratified 70/15/15 split. Before model development, the datasets are validated to confirm their dimensions, feature consistency, target variable, and data quality.

In [3]:
print("Loading training dataset...")
train_df = pd.read_csv(TRAIN_FILE, low_memory=False)

print("Loading validation dataset...")
val_df = pd.read_csv(VAL_FILE, low_memory=False)

print("\nDatasets loaded successfully.")
print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Loading training dataset...
Loading validation dataset...

Datasets loaded successfully.
Training shape: (1764525, 53)
Validation shape: (378113, 53)


In [4]:
TARGET = "Attack Type"

print("Target column present in training:", TARGET in train_df.columns)
print("Target column present in validation:", TARGET in val_df.columns)

print("\nSame columns in both datasets:",
      train_df.columns.tolist() == val_df.columns.tolist())

print("\nNumber of training features:", len(train_df.columns) - 1)
print("Number of validation features:", len(val_df.columns) - 1)

print("\nTraining target classes:")
print(train_df[TARGET].value_counts())

Target column present in training: True
Target column present in validation: True

Same columns in both datasets: True

Number of training features: 52
Number of validation features: 52

Training target classes:
Attack Type
Normal Traffic    1466539
DoS                135621
DDoS                89610
Port Scanning       63486
Brute Force          6405
Web Attacks          1500
Bots                 1364
Name: count, dtype: int64


### Pre-Model Data Quality Check

Before training the baseline models, the training and validation datasets are checked for missing values, infinite numeric values, duplicate records, and unexpected non-numeric model features. These checks help ensure that model performance is not affected by unresolved preprocessing issues.

In [5]:
# Missing values
train_missing = train_df.isna().sum().sum()
val_missing = val_df.isna().sum().sum()

# Separate model features from target
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

# Check feature data types
non_numeric_train = X_train.select_dtypes(exclude=np.number).columns.tolist()
non_numeric_val = X_val.select_dtypes(exclude=np.number).columns.tolist()

# Check infinity only in numeric model features
train_inf = np.isinf(X_train.select_dtypes(include=np.number)).sum().sum()
val_inf = np.isinf(X_val.select_dtypes(include=np.number)).sum().sum()

# Duplicate rows
train_duplicates = train_df.duplicated().sum()
val_duplicates = val_df.duplicated().sum()

print("PRE-MODEL DATA QUALITY CHECK")
print("-" * 35)

print("Training missing values:", train_missing)
print("Validation missing values:", val_missing)

print("\nTraining infinite values:", train_inf)
print("Validation infinite values:", val_inf)

print("\nTraining duplicate rows:", train_duplicates)
print("Validation duplicate rows:", val_duplicates)

print("\nNon-numeric training features:", non_numeric_train)
print("Non-numeric validation features:", non_numeric_val)

PRE-MODEL DATA QUALITY CHECK
-----------------------------------
Training missing values: 0
Validation missing values: 0

Training infinite values: 0
Validation infinite values: 0

Training duplicate rows: 82
Validation duplicate rows: 4

Non-numeric training features: []
Non-numeric validation features: []


### Duplicate and Split Leakage Review

A small number of duplicate records remain within the processed training and validation datasets. Because the datasets were created using a random stratified split, an additional check is performed to determine whether identical feature records appear in both the training and validation sets.

Cross-split duplicates may introduce data leakage because the model could be evaluated on observations identical to records it encountered during training.

In [6]:
# Create unique feature-only records for the overlap check
train_unique = X_train.drop_duplicates()
val_unique = X_val.drop_duplicates()

# Find identical feature rows appearing in both datasets
overlap = train_unique.merge(
    val_unique,
    how="inner",
    on=X_train.columns.tolist()
)

print("Unique training feature rows:", len(train_unique))
print("Unique validation feature rows:", len(val_unique))
print("Identical feature rows across train and validation:", len(overlap))

Unique training feature rows: 1764095
Unique validation feature rows: 378094
Identical feature rows across train and validation: 190
